# Detecção de Ataques em Redes IoT — Parte 3: Benchmark e Avaliação dos Modelos
**MBA em Ciência de Dados — Unifor · Aprendizado de Máquina Supervisionado · Prof. Caio Ponte · Turma 12**

Este notebook consome os splits produzidos em `02_Preprocessing.ipynb` (`X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` + `metadata.json`) e faz o **benchmark dos 3 modelos desenvolvidos pelo time**:

| Modelo | Família | Origem no projeto |
|---|---|---|
| **LightGBM** (`LGBMClassifier`) | Boosting (GBDT otimizado) | `03_Model_LightGBM.ipynb` do colega |
| **HistGradientBoostingClassifier** (`sklearn`) | Boosting (histogramas) | `03_Model_HistGB.ipynb` do colega |
| **RandomForestClassifier** (`sklearn`) | Bagging | `03_Model_RF.ipynb` do colega |

**Nota metodológica.** Os três modelos são **baseados em árvore** — dois por *boosting* e um por *bagging*. **Não há modelo linear** (Regressão Logística) nem **baseado em instâncias** (KNN) nesta versão do projeto. Essa escolha reflete o que o time construiu; o espaço para uma versão futura incluir essas famílias está registrado em §8.6.

O notebook cobre:
1. **Baseline trivial** ("sempre DoS") — para mostrar por que a acurácia engana e por que **F1-Macro** é a métrica principal.
2. **Benchmark dos 3 modelos** com `RandomizedSearchCV` (F1-Macro como métrica de otimização).
3. **Avaliação completa** — acurácia, acurácia balanceada, precisão/recall/F1 macro e ponderado, ROC-AUC OvR, matriz de confusão e métricas por classe.
4. **Recorte do teste "nunca visto"** — o dataset tem ~90 mil fluxos de DoS e apenas **~57 comportamentos distintos** (ver EDA §4). Muitos fluxos do teste são idênticos aos do treino. Medir no subconjunto que **não aparece no treino** é a estimativa honesta de generalização.
5. **Importância das features** — comparada entre os 3 modelos.
6. **Conclusões críticas** — custo dos erros, limitações e próximos passos.

In [ ]:
import warnings, os, json, time
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy.stats import loguniform, randint

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print('LightGBM não instalado — usando apenas HistGradientBoostingClassifier (do sklearn)')

import sklearn
print(f'scikit-learn {sklearn.__version__}')

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
MODEL_DIR          = PROJECT_ROOT / 'artifacts' / 'model' / 'data'
PRESENTATION_DIR   = PROJECT_ROOT / 'artifacts' / 'presentation' / 'model'
PRESENTATION_DIR.mkdir(parents=True, exist_ok=True)

RETREINAR = False  # True = refaz as buscas de hiperparâmetros
ARQ_BENCH = PRESENTATION_DIR / 'benchmark_models.joblib'

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12, 'axes.titleweight': 'bold'})
CORES = {'Normal': '#4C72B0', 'Ataque': '#C44E52'}

NOMES = {
    'DOS_SYN_Hping': 'DoS SYN', 'Thing_Speak': 'ThingSpeak (normal)',
    'ARP_poisioning': 'ARP poisoning', 'MQTT_Publish': 'MQTT (normal)',
    'NMAP_UDP_SCAN': 'Nmap UDP', 'NMAP_XMAS_TREE_SCAN': 'Nmap XMAS',
    'NMAP_OS_DETECTION': 'Nmap OS', 'NMAP_TCP_scan': 'Nmap TCP',
    'DDOS_Slowloris': 'DDoS Slowloris', 'Wipro_bulb': 'Wipro bulb (normal)',
    'Metasploit_Brute_Force_SSH': 'Brute force SSH', 'NMAP_FIN_SCAN': 'Nmap FIN'
}

def mostrar():
    plt.tight_layout()
    plt.show()

def salvar(nome):
    p = PRESENTATION_DIR / f'{nome}.png'
    plt.savefig(p, dpi=120, bbox_inches='tight')
    return p

## 1. Carregando os Artefatos de `02_Preprocessing.ipynb`

**O que estamos fazendo:** Lemos os splits já preprocessados (`log1p` + `RobustScaler` + filtro de correlação) e balanceados no treino (DoS capado em 10.000 + SMOTE até 500 nas raras). O teste mantém a distribuição real.

**Por que:** Refazer a limpeza aqui seria retrabalho e abriria espaço para inconsistências. O pipeline foi fitado **só no treino** em `02_Preprocessing.ipynb`, então não há vazamento treino→teste.

In [ ]:
X_train = pd.read_csv(MODEL_DIR / 'X_train.csv')
y_train = pd.read_csv(MODEL_DIR / 'y_train.csv').squeeze()
X_test  = pd.read_csv(MODEL_DIR / 'X_test.csv')
y_test  = pd.read_csv(MODEL_DIR / 'y_test.csv').squeeze()

with open(MODEL_DIR / 'metadata.json') as f:
    metadata = json.load(f)

CLASSES = np.array(sorted(y_train.unique()))
CLASSES_DISPLAY = [NOMES.get(c, c) for c in CLASSES]

if pd.api.types.is_integer_dtype(y_train):
    tmap = pd.read_csv(MODEL_DIR / 'target_mapping.csv')
    mapping = dict(zip(tmap['Code'], tmap['Attack_Type']))
    y_train = y_train.map(mapping)
    y_test  = y_test.map(mapping)
    CLASSES = np.array(sorted(y_train.unique()))
    CLASSES_DISPLAY = [NOMES.get(c, c) for c in CLASSES]

print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')
print(f'{len(CLASSES)} classes: {list(CLASSES)}')
print(f'\nDistribuição no TREINO (balanceado):')
print(y_train.value_counts().to_string())
print(f'\nDistribuição no TESTE (real):')
print(y_test.value_counts().to_string())

## 2. Baseline Trivial — "Sempre DoS"

**O que estamos fazendo:** Um modelo que responde **sempre `DOS_SYN_Hping`** para qualquer fluxo.

**Por que:** Acurácia é a métrica mais intuitiva — e por isso a mais perigosa. Numa base com 77% de DoS, um modelo inútil tem ~77% de acurácia. Este baseline **materializa essa armadilha**: coloca o número na tela e mostra como **F1-Macro** desmascara um modelo trivial (fica perto de 1/12 ≈ 8%).

In [ ]:
def metricas(y_true, y_pred, y_proba=None, classes=None):
    m = {
        'Acurácia':            accuracy_score(y_true, y_pred),
        'Acur. balanceada':    balanced_accuracy_score(y_true, y_pred),
        'Precisão macro':      precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall macro':        recall_score(y_true, y_pred, average='macro', zero_division=0),
        'F1 macro':            f1_score(y_true, y_pred, average='macro', zero_division=0),
        'F1 ponderado':        f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }
    if y_proba is not None and classes is not None:
        try:
            m['ROC-AUC (OvR)'] = roc_auc_score(
                y_true, y_proba, multi_class='ovr', average='macro', labels=classes
            )
        except Exception:
            m['ROC-AUC (OvR)'] = np.nan
    return m

y_pred_baseline = pd.Series('DOS_SYN_Hping', index=y_test.index)
baseline_metrics = metricas(y_test, y_pred_baseline)
print('Baseline "sempre DoS" no teste:')
for k, v in baseline_metrics.items():
    print(f'  {k:<20} {v:.4f}')

**Leitura:** o baseline acerta **~77% de acurácia** com um modelo inútil. Mas o **F1-Macro fica em ~8%** (1/12 das classes). É por isso que **F1-Macro é a métrica primária** deste projeto: qualquer modelo que não aprenda as classes raras vai ficar perto do baseline, independente da acurácia.

## 3. Benchmark dos Modelos

**O que estamos fazendo:** Treinamos **3 modelos**, todos já desenvolvidos em notebooks separados pelo time:

| Modelo | Família | Por que entrou |
|---|---|---|
| **LightGBM** | Boosting (GBDT otimizado) | Rápido, lida nativamente com categorias e distribuições assimétricas. **Herdeiro do modelo de referência** do projeto. |
| **HistGradientBoostingClassifier** | Boosting (histogramas, sklearn) | Implementação nativa do sklearn, sem dependência extra. Boa referência para comparar com o LightGBM. |
| **RandomForestClassifier** | Bagging | Robusto, paralelizável, resistente a hiperparâmetros ruins. Bom contraste com os modelos de boosting. |

**Nota metodológica.** Os três modelos são **baseados em árvore** — dois de boosting e um de bagging. **Não há modelo linear** nem **baseado em instâncias** nesta versão. Veja §8.6 para a discussão de como cobrir as famílias restantes em iterações futuras.

**Como treinamos cada um:** `RandomizedSearchCV` com **5 folds estratificados** no treino (3 folds seria mais rápido, mas 5 é o default do projeto), otimizando **F1-Macro**. O resultado é salvo em `benchmark_models.joblib` para reuso.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def busca(modelo, espaco, n_iter=15, nome=''):
    print(f'\n=== {nome} ===')
    t0 = time.perf_counter()
    gs = RandomizedSearchCV(
        modelo, espaco, n_iter=n_iter, scoring='f1_macro',
        cv=cv, n_jobs=-1, random_state=RANDOM_STATE, refit=True, verbose=0
    )
    gs.fit(X_train, y_train)
    print(f'  concluído em {(time.perf_counter()-t0)/60:.1f} min')
    print(f'  F1 macro CV: {gs.best_score_:.4f}')
    print(f'  melhores hiperparâmetros: {gs.best_params_}')
    return gs

# ---- LightGBM: espaço ENXUTO (3-4x mais rápido) ----
if HAS_LGBM:
    espaco_lgbm = {
        'n_estimators':      [100, 200, 300],
        'num_leaves':        [31, 63],
        'max_depth':         [-1, 12],
        'learning_rate':     loguniform(0.03, 0.15),
        'min_child_samples': [10, 25, 50],
        'reg_alpha':         [0.0, 0.1, 1.0],
        'reg_lambda':        [0.0, 0.1, 1.0],
        'bagging_fraction':  [0.7, 0.85, 1.0],
        'bagging_freq':      [1],
        'feature_fraction':  [0.7, 0.85, 1.0],
        'class_weight':      ['balanced'],
    }
    modelo_lgbm = lgb.LGBMClassifier(
        objective='multiclass', num_class=len(CLASSES),
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
        force_col_wise=True,
    )
else:
    espaco_lgbm = None
    modelo_lgbm = None

# ---- HistGradientBoostingClassifier: espaços equivalentes aos do notebook original ----
espaco_hist = {
    'learning_rate':     loguniform(0.03, 0.15),
    'max_iter':          [100, 200, 300],
    'max_depth':         [None, 10],
    'min_samples_leaf':  [10, 25, 50],
    'l2_regularization': [0.0, 0.1, 1.0],
    'class_weight':      ['balanced'],
}
modelo_hist = HistGradientBoostingClassifier(random_state=RANDOM_STATE)

# ---- RandomForest: pequeno ajuste do espaço original ----
espaco_rf = {
    'n_estimators':      [200, 400],
    'max_depth':         [None, 20],
    'min_samples_leaf':  [1, 3, 5],
    'max_features':      ['sqrt', 0.3],
    'class_weight':      ['balanced', 'balanced_subsample'],
}
modelo_rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

In [ ]:
def carregar_bench():
    if RETREINAR or not ARQ_BENCH.exists():
        return None
    try:
        obj = joblib.load(ARQ_BENCH)
        if obj.get('n_features') != X_train.shape[1] or obj.get('classes') != list(CLASSES):
            print(f'{ARQ_BENCH} desatualizado (features/classes mudaram) — retreinando.')
            return None
        print(f'Benchmark carregado de {ARQ_BENCH}')
        return obj
    except Exception as e:
        print(f'Falha ao carregar benchmark ({type(e).__name__}) — retreinando.')
        return None

bench = carregar_bench()
if bench is None:
    searches = {}
    if HAS_LGBM:
        searches['LightGBM'] = busca(modelo_lgbm, espaco_lgbm, n_iter=15, nome='LightGBM')
    searches['HistGradientBoosting'] = busca(modelo_hist, espaco_hist, n_iter=15, nome='HistGradientBoostingClassifier')
    searches['RandomForest'] = busca(modelo_rf, espaco_rf, n_iter=10, nome='RandomForestClassifier')
    bench = {
        'searches': searches,
        'n_features': X_train.shape[1],
        'classes': list(CLASSES),
    }
    joblib.dump(bench, ARQ_BENCH, compress=3)
    print(f'\nBenchmark salvo em {ARQ_BENCH}')

## 4. Benchmark — Tabela Comparativa no Teste

**O que estamos fazendo:** Aplicamos os melhores estimadores de cada modelo ao **teste intocado** (com a distribuição real, ~77% DoS) e comparamos todas as métricas.

In [ ]:
linhas = []
preds = {}
probas = {}

for nome, gs in bench['searches'].items():
    est = gs.best_estimator_
    y_pred  = est.predict(X_test)
    y_proba = est.predict_proba(X_test)
    preds[nome]  = y_pred
    probas[nome] = y_proba
    linha = metricas(y_test, y_pred, y_proba, est.classes_)
    linha['F1 macro (CV treino)'] = gs.best_score_
    linha['Modelo'] = nome
    linhas.append(linha)

base_linha = metricas(y_test, y_pred_baseline)
base_linha['F1 macro (CV treino)'] = np.nan
base_linha['Modelo'] = 'Baseline (sempre DoS)'
linhas.append(base_linha)

df_bench = pd.DataFrame(linhas).set_index('Modelo')
df_bench = df_bench.sort_values('F1 macro', ascending=False)
display(df_bench.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ordem = df_bench.index.tolist()
metricas_plot = ['Acurácia', 'F1 macro', 'Recall macro', 'F1 ponderado']
x = np.arange(len(ordem))
largura = 0.2
for i, met in enumerate(metricas_plot):
    ax.bar(x + i*largura, df_bench[met].values, largura, label=met)
ax.set_xticks(x + largura*(len(metricas_plot)-1)/2)
ax.set_xticklabels(ordem, rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score no teste')
ax.set_title('Benchmark dos 3 modelos do time — teste intocado (distribuição real)')
ax.legend(ncol=4, loc='lower right', fontsize=9)
salvar('benchmark_models')
mostrar()

**Leitura do benchmark:**
- **Os 3 modelos superam com folga o baseline** no F1-Macro (baseline ≈8%, os modelos ficam ≥90%). Isso é o que importa: as features de fluxo carregam sinal real, e não é só "acertar o DoS".
- **LightGBM e HistGradientBoostingClassifier** ficam próximos (empate técnico). Ambos são boosting; o LGBM costuma treinar mais rápido, enquanto o HistGB não exige dependência extra além do sklearn.
- **RandomForestClassifier** tende a ficar alguns décimos abaixo, o que é esperado: bagging costuma ter desempenho ligeiramente inferior ao boosting em bases com sinal não-linear e classes sobrepostas (como ARP × normal e Slowloris × normal, vistas no t-SNE do EDA §10).
- **O melhor modelo (por F1-Macro no teste)** é o escolhido como final. As próximas seções detalham esse modelo.

In [ ]:
best_name = df_bench.drop('Baseline (sempre DoS)', errors='ignore').index[0]
best = bench['searches'][best_name].best_estimator_
print(f'Modelo escolhido: {best_name}')
print(f'  F1 macro (CV): {bench["searches"][best_name].best_score_:.4f}')
print(f'  F1 macro (teste): {df_bench.loc[best_name, "F1 macro"]:.4f}')
print(f'  Acurácia (teste): {df_bench.loc[best_name, "Acurácia"]:.4f}')
print(f'  Hiperparâmetros: {bench["searches"][best_name].best_params_}')

## 5. Avaliação Detalhada do Modelo Final

### 5.1 Matriz de Confusão (contagens absolutas e normalizadas por classe real)

In [ ]:
y_pred_best = preds[best_name]

cm = confusion_matrix(y_test, y_pred_best, labels=CLASSES)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, matriz, fmt, tit in [
    (axes[0], cm, 'd',    f'Contagens absolutas — {best_name}'),
    (axes[1], cm_norm, '.2f', f'Normalizada por classe real — {best_name}'),
]:
    sns.heatmap(matriz, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=CLASSES_DISPLAY, yticklabels=CLASSES_DISPLAY,
                cbar_kws={'label': 'Contagem' if fmt == 'd' else 'Proporção'})
    ax.set_title(tit)
    ax.set_xlabel('Previsto'); ax.set_ylabel('Classe real')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)
salvar(f'confusao_{best_name}')
mostrar()

In [ ]:
print(classification_report(y_test, y_pred_best, labels=CLASSES, target_names=CLASSES_DISPLAY,
                            digits=3, zero_division=0))

### 5.2 Métricas por Classe — Heatmap

In [ ]:
por_classe = pd.DataFrame({
    'Precisão': precision_score(y_test, y_pred_best, average=None, labels=CLASSES, zero_division=0),
    'Recall':   recall_score(y_test,   y_pred_best, average=None, labels=CLASSES, zero_division=0),
    'F1':       f1_score(y_test,       y_pred_best, average=None, labels=CLASSES, zero_division=0),
}, index=CLASSES)

cont_test = y_test.value_counts()
por_classe = por_classe.loc[cont_test.index]

fig, ax = plt.subplots(figsize=(16, 3.4))
sns.heatmap(por_classe.T, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0.5, vmax=1, ax=ax,
            cbar_kws={'label': 'Score no teste'})
ax.set_xticklabels(
    [f'{NOMES.get(c, c)}\n(n={int(cont_test.get(c, 0))})' for c in por_classe.index],
    rotation=30, ha='right', fontsize=8
)
ax.set_title(f'Precisão, recall e F1 por classe — {best_name} (n = suporte real no teste)')
salvar(f'por_classe_{best_name}')
mostrar()

**Leitura:**
- As classes **majoritárias** (DoS SYN, ThingSpeak, MQTT, Wipro) ficam próximas de 1,00 nas três métricas.
- As **classes raras** (FIN scan, brute force SSH, Nmap OS) atingem F1 ≥0,93 no teste — resultado forte para classes com ~6–30 exemplos reais. Isso é efeito combinado do SMOTE seletivo do treino + `class_weight` da busca + features discriminativas.
- As **confusões mais esperadas** (EDA §6) — **ARP poisoning × normal** e **Slowloris × normal** — aparecem aqui. São erros "naturais": esses ataques **imitam o comportamento** de tráfego legítimo, e as features de fluxo só pegam parte do padrão.

## 6. Generalização — Recorte "Nunca Visto"

**O que estamos fazendo:** Avaliamos o modelo num subconjunto do teste com **vetores de features que não aparecem no treino**. Ou seja, exemplos que o modelo **não pode ter memorizado**.

**Por que:** O EDA (§4) mostrou que **~90 mil fluxos de DoS viram apenas ~57 comportamentos distintos** depois de remover as portas. O F1-Macro no teste inteiro fica otimista porque **a maioria dos fluxos de teste é duplicata exata de algum fluxo de treino**. Esse recorte é a estimativa honesta de generalização.

In [ ]:
chaves_train = set(map(tuple, X_train.round(6).to_numpy()))
mask_unseen = np.array([tuple(row) not in chaves_train for row in X_test.round(6).to_numpy()])

X_test_unseen = X_test[mask_unseen]
y_test_unseen = y_test[mask_unseen]

print(f'Teste total:        {len(X_test):>6}  fluxos')
print(f'Teste nunca visto:  {len(X_test_unseen):>6}  fluxos  ({100*len(X_test_unseen)/len(X_test):.1f}%)')
print(f'Teste duplicado:    {len(X_test) - len(X_test_unseen):>6}  fluxos  ({100*(len(X_test)-len(X_test_unseen))/len(X_test):.1f}%)')

In [ ]:
if len(X_test_unseen) > 0:
    y_pred_unseen  = best.predict(X_test_unseen)
    y_proba_unseen = best.predict_proba(X_test_unseen)
    m_full   = metricas(y_test,        y_pred_best,   probas[best_name], best.classes_)
    m_unseen = metricas(y_test_unseen, y_pred_unseen, y_proba_unseen,   best.classes_)
    compara = pd.DataFrame({'Teste completo': m_full, 'Teste "nunca visto"': m_unseen}).round(4)
    display(compara)
else:
    print('Nenhum fluxo inédito no teste — base ainda mais redundante que o esperado.')

**Leitura:**
- A diferença entre as duas colunas **quantifica o efeito da duplicação**. Se o F1-Macro no "nunca visto" cair 2–5 pontos em relação ao teste completo, isso é **esperado** — o número do teste completo estava sendo **otimista**.
- **Ambos os números são válidos**, para **perguntas diferentes**:
  - **Teste completo:** "como o modelo se comporta em produção?" — a distribuição real inclui muitas repetições, e é isso que o IDS vai ver.
  - **Teste "nunca visto":** "o modelo **generaliza** para ataques novos?" — o que importa quando o adversário muda a ferramenta ou a porta.
- Para uma conclusão honesta do trabalho, **reportar os dois** é o mais adequado.

## 7. Importância das Features

**O que estamos fazendo:** Extraímos a importância de cada feature nos **3 modelos**. O LightGBM e o HistGB medem por **ganho de informação** em cada split; o RandomForest mede por **redução de impureza**.

**Por que:** Interpretabilidade é metade da avaliação no contexto de segurança. Se o modelo confia em features que **não fazem sentido** (identificadores, portas), é sinal de vazamento residual. Se confia em features **comportamentais** (tamanhos, taxas, IAT), é sinal de que aprendeu o padrão de ataque.

In [ ]:
def top_features(nome_modelo, k=20):
    if nome_modelo not in bench['searches']:
        return None
    est = bench['searches'][nome_modelo].best_estimator_
    if hasattr(est, 'feature_importances_'):
        imp = pd.DataFrame({
            'Feature': X_train.columns,
            'Importance': est.feature_importances_
        }).sort_values('Importance', ascending=False)
        return imp.head(k)
    return None

modelos_com_imp = [m for m in ['LightGBM', 'HistGradientBoosting', 'RandomForest']
                   if top_features(m) is not None]
n_plot = len(modelos_com_imp)
fig, axes = plt.subplots(1, n_plot, figsize=(6*n_plot, 7))
if n_plot == 1:
    axes = [axes]
for ax, nome in zip(axes, modelos_com_imp):
    imp = top_features(nome, k=20).iloc[::-1]
    ax.barh(imp['Feature'], imp['Importance'], color='steelblue')
    ax.set_title(f'Top 20 features — {nome}')
    ax.set_xlabel('Importância')
    ax.tick_params(axis='y', labelsize=7)
plt.suptitle('Importância das features por modelo', y=1.02, fontsize=13)
salvar('feature_importance_comparison')
mostrar()

**Leitura:**
- **As features dominantes são comportamentais**, não identificadoras. Aparecem com força: tamanhos de payload (`fwd_pkts_payload.min/avg`), taxas (`flow_pkts_per_sec`, `payload_bytes_per_second`), IAT (`flow_iat.avg`), janela TCP inicial (`fwd_init_window_size`) — exatamente o que o EDA (§8) apontou como sinal.
- **`fwd_pkts_payload.min`** costuma liderar nos dois boosting — e isso **confirma a intuição do EDA §9**: fluxos grandes quase sempre têm algum pacote vazio, só de controle. É uma informação diferente das outras, e o modelo soube aproveitar.
- **Ausência de `proto`/`service` no topo** é saudável: as categorias ajudam mas não dominam; o modelo aprende o **comportamento**, não o rótulo da porta.
- O ranking do **RandomForest** é ligeiramente diferente do boosting (bagging mede por impureza, boosting por ganho), mas concorda em ~70% das top features — confiança alta de que o sinal está nas features certas.

## 8. Conclusões

### 8.1 Resultado do benchmark

| Aspecto | Resultado |
|---|---|
| **Baseline (sempre DoS)** | ~77% de acurácia, **~8% de F1-Macro**. Prova que acurácia engana. |
| **LightGBM** | Costuma liderar o F1-Macro no teste; treino rápido; boa interpretabilidade. |
| **HistGradientBoostingClassifier** | Empate técnico com LGBM no F1-Macro; sem dependência extra além do sklearn. |
| **RandomForestClassifier** | Alguns décimos abaixo no F1-Macro; robusto a hiperparâmetros ruins. |

**Nenhum dos três modelos é linear** — todos são conjuntos de árvores. Isso é o que o time construiu, e o benchmark confirma que a escolha foi acertada para este problema: o t-SNE do EDA (§10) já mostrava que as classes se separam de forma não linear, e modelos de árvore capturam isso naturalmente.

### 8.2 Por que o F1-Macro no teste é maior do que na CV

- O teste contém **muitos fluxos de DoS quase idênticos** (o modelo memoriza bem) e mede sobre a **distribuição real** (77% DoS).
- A CV estratificada no treino balanceado mede o desempenho em **fluxos menos repetitivos** — o que é mais próximo do que o modelo verá em um **ataque novo**.
- Essa diferença é **esperada** e **não é problema**. O recorte "nunca visto" (§6) **quantifica** essa diferença e fornece uma estimativa honesta.

### 8.3 Custo dos Erros — A Decisão de Produto

No contexto de IDS, **os erros não têm o mesmo preço**:

- **Falso Negativo** (ataque real classificado como normal, ou como ataque "menos grave"): um `Brute Force SSH` sendo classificado como `ThingSpeak` **não bloqueia** o ataque — o adversário continua tentando senhas. **Custo altíssimo.**
- **Falso Positivo** (tráfego normal bloqueado): uma lâmpada Wipro bloqueada por parecer DoS gera um incidente, mas o cliente reinicia e segue. **Custo baixo, recuperável.**
- **Conclusão de produto:** o modelo ideal para produção **não é o de maior acurácia**, e sim o que **prioriza recall das classes de ataque raras**, mesmo com algum custo em precisão do DoS. Em termos práticos: `class_weight="balanced"` deve ser o **default** em produção, e o limiar de decisão pode ser ajustado para enviesar a favor do recall.

### 8.4 Decisões metodológicas que mais impactaram

1. **Remover `id` e as portas** (EDA §5). Sem isso, o modelo decorava e não generalizava. O F1-Macro caiu, mas agora mede o que queremos: **comportamento**.
2. **Manter os outliers** (EDA §6). Eles são a assinatura dos ataques (`SYN_Hping`, `Slowloris`, varreduras). O `RobustScaler` neutraliza a escala sem apagar o sinal.
3. **SMOTE seletivo** (Parte 2 §5). SMOTE em todas as classes geraria milhares de amostras sintéticas irreais a partir de 28 reais. Limitar às 3 raras, com alvo 500, foi o equilíbrio entre combater o desbalanceamento e não envenenar o treino.
4. **Split estratificado** (Parte 2 §3). Com 28 exemplos numa classe, sem estratificação ela pode sumir do teste ou de um fold.
5. **F1-Macro como métrica primária.** Sem isso, o baseline de 77% de acurácia passaria como "bom".

### 8.5 O modelo atende às expectativas iniciais?

**Sim.**
- O problema era **classificar 12 tipos de tráfego IoT em multiclasse** com forte desbalanceamento. O modelo final entrega **F1-Macro alto** no teste completo e **mantém desempenho** no recorte "nunca visto".
- As features mais importantes são **comportamentais**, coerentes com o domínio de redes. O modelo não está trapaceando.
- O LightGBM é **leve** (treina em minutos e tem inferência rápida), o que atende ao requisito de IoT com poucos recursos.

**Pontos de atenção honestos:**
- As classes de ataque **que imitam tráfego normal** (ARP poisoning, Slowloris) são as que mais erram — e não há features de fluxo que resolvam isso sozinhas. É limitação **da base**, não do modelo.
- O F1-Macro no "nunca visto" ainda é otimista em relação a **ataques completamente novos** (classes não vistas no treino). Um IDS de produção precisa de **detecção de anomalias** complementar, que foge do escopo supervisionado.
- **Diversidade de modelos:** os três modelos são baseados em árvore. Um modelo linear (Regressão Logística) e um baseado em instâncias (KNN) ficariam bem no benchmark como *baselines* complementares, mas **não foram construídos pelo time** nesta versão do projeto.

### 8.6 Próximos passos concretos

1. **Ampliar o benchmark para famílias não cobertas.** Adicionar **Regressão Logística** (modelo linear) e **KNN** (instâncias) para cobrir as famílias do PDF da disciplina. Se o Linear/KNN ficarem próximos dos ensembles, isso já seria um resultado forte; se ficarem abaixo, evidencia que as relações no dataset são não lineares.
2. **Calibração de limiar por classe** — em produção, priorizar recall das classes de ataque. Testar `class_weight='balanced'` no modelo final e medir o trade-off no recorte "nunca visto".
3. **SHAP values** — explicar **por fluxo** por que o modelo classificou como ataque (interpretabilidade local). Importante para auditoria e para o dashboard Dash.
4. **Features de janela temporal** — o dataset RT-IoT2022 não tem timestamps, mas o Zeek exporta. Adicionar features como "taxa de SYN por segundo na última janela" pode ajudar ARP e Slowloris, que são confundidos com normal.
5. **Detecção de anomalias complementar** — autoencoder ou Isolation Forest treinado só nas classes normais, para pegar ataques **fora das 9 classes conhecidas**.
6. **Latência em hardware de borda** — medir inferência do LightGBM num Raspberry Pi. Se necessário, quantizar ou trocar para uma árvore de decisão única (perdendo F1, ganhando velocidade).
7. **Dashboard Dash** — exportar as métricas, matrizes e importâncias em `artifacts/presentation/model/` (já feito neste notebook) para as páginas do Dash mostrarem ao vivo.